# GDACS — Historical National Exposure

Retrieves country-level population exposure for all tropical cyclones within a date range, using the GDACS search endpoint for full historical coverage. Saves national level exposure estimates to blob storage for all available storms.

**API path:**
```
geteventlist/search  (paginated, filterable by date / source)
  -> getepisodedata  (latest episode per event)
    -> getimpact -> datums[alias='country'] -> ISO_3DIGIT, CNTRY_NAME, POP_AFFECTED
```

In [15]:
import requests
import pandas as pd
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

GDACS_BASE = 'https://www.gdacs.org/gdacsapi/api'
FROM_DATE = '2010-01-01'
TO_DATE   = '2026-12-31'
SOURCE    = 'NOAA'   # 'NOAA' = Atlantic/E.Pacific | 'JTWC' = W.Pacific/Indian Ocean
OUTPUT_CSV = 'gdacs_historical_national_exposure.csv'

# Wind speed (kt) implied by each buffer key
# Have spot-checked throughout, and there only ever seems to be these two buffers
BUFFER_KT = {
    'buffer39': 34,
    'buffer74': 64,
}

## 1. Fetch all TC events in the date range

In [3]:
all_events = []
page = 1

while True:
    params = {
        'eventlist':  'TC',
        'fromDate':   FROM_DATE,
        'toDate':     TO_DATE,
        'pageSize':   100,
        'pageNumber': page,
    }
    resp = requests.get(
        f'{GDACS_BASE}/events/geteventlist/search',
        params=params,
        timeout=30,
    )
    # API returns an empty body (not JSON) when there are no more pages
    if not resp.text.strip():
        break
    features = resp.json().get('features', [])
    if not features:
        break

    for f in features:
        p = f['properties']
        if SOURCE and p.get('source') != SOURCE:
            continue
        all_events.append({
            'event_id':    str(p['eventid']),
            'storm_name':  p.get('name', ''),
            'source':      p.get('source', ''),
            'from_date':   p.get('fromdate', ''),
            'to_date':     p.get('todate', ''),
            'alert_level': p.get('alertlevel', ''),
        })

    print(f'Page {page}: {len(features)} events fetched')
    page += 1

df_events = pd.DataFrame(all_events)
print(f'\nTotal TCs found ({SOURCE or "all basins"}): {len(df_events)}')

Page 1: 100 events fetched
Page 2: 100 events fetched
Page 3: 90 events fetched

Total TCs found (NOAA): 52


In [5]:
df_events.head()

,event_id,storm_name,source,from_date,to_date,alert_level
0,1001230,Tropical Cyclone MELISSA-25,NOAA,2025-10-21T15:00:00,2025-10-31T15:00:00,Red
1,1001168,Tropical Cyclone ERICK-25,NOAA,2025-06-16T21:00:00,2025-06-20T03:00:00,Red
2,1001122,Tropical Cyclone RAFAEL-24,NOAA,2024-11-03T21:00:00,2024-11-10T21:00:00,Orange
3,1001114,Tropical Cyclone OSCAR-24,NOAA,2024-10-19T15:00:00,2024-10-22T18:00:00,Orange
4,1001113,Tropical Cyclone NADINE-24,NOAA,2024-10-18T21:00:00,2024-10-20T15:00:00,Orange


## 2. For each event, fetch exposure data

From the last available episode.

In [6]:
def fetch_national_exposure(event_id):
    '''Return country-row dicts for the latest episode. Empty list if no data.'''
    try:
        props = requests.get(
            f'{GDACS_BASE}/events/getepisodedata',
            params={'eventtype': 'TC', 'eventid': event_id},
            timeout=30,
        ).json()['properties']
    except Exception as e:
        print(f'  [{event_id}] failed: {e}')
        return []

    last_ep_url = props.get('episodes', [{}])[-1].get('details', '')
    episode_id  = last_ep_url.split('episodeid=')[-1].split('&')[0] if last_ep_url else '?'
    buffers     = {k: v for k, v in props.get('impacts', [{}])[0].get('resource', {}).items()
                   if k.startswith('buffer')}

    country_data = {}
    for buf, url in buffers.items():
        col = f'pop_{BUFFER_KT.get(buf, buf)}kt'
        try:
            datums = requests.get(url, timeout=30).json().get('datums', [])
        except Exception:
            continue
        country_datum = next((d for d in datums if d['alias'] == 'country'), None)
        if not country_datum:
            continue
        for row in country_datum.get('datum', []):
            sc   = {s['name']: s['value'] for s in row['scalars']['scalar']}
            iso3 = sc.get('ISO_3DIGIT')
            if not iso3:
                continue
            # NOTE: Some storms have POP_AFFECTED_TEMP, which seems to consistently have a value of 0.
            # We ignore the 0 in those cases and only populate the column if POP_AFFECTED is present and non-zero.
            country_data.setdefault(iso3, {
                'event_id': event_id, 'episode_id': episode_id,
                'iso3': iso3, 'country_name': sc.get('CNTRY_NAME'),
            })[col] = int(sc['POP_AFFECTED']) if sc.get('POP_AFFECTED') else None

    return list(country_data.values())

In [7]:
all_rows = []

for _, ev in df_events.iterrows():
    eid  = ev['event_id']
    name = ev['storm_name']
    print(f'Fetching {name} ({eid}) …', end=' ')
    rows = fetch_national_exposure(eid)
    for r in rows:
        r['storm_name']  = name
        r['source']      = ev['source']
        r['from_date']   = ev['from_date']
        r['alert_level'] = ev['alert_level']
    all_rows.extend(rows)
    print(f'{len(rows)} countries')

df = (
    pd.DataFrame(all_rows)
    .sort_values(['from_date', 'storm_name'], ascending=False)
    .reset_index(drop=True)
)

# Reorder columns
leading  = ['storm_name', 'event_id', 'episode_id', 'source', 'from_date', 'alert_level',
            'iso3', 'country_name']
pop_cols = sorted([c for c in df.columns if c.startswith('pop_')])
df = df[leading + pop_cols]

Fetching Tropical Cyclone MELISSA-25 (1001230) … 9 countries
Fetching Tropical Cyclone ERICK-25 (1001168) … 1 countries
Fetching Tropical Cyclone RAFAEL-24 (1001122) … 3 countries
Fetching Tropical Cyclone OSCAR-24 (1001114) … 4 countries
Fetching Tropical Cyclone NADINE-24 (1001113) … 5 countries
Fetching Tropical Cyclone MILTON-24 (1001111) … 4 countries
Fetching Tropical Cyclone JOHN-24 (1001100) … 1 countries
Fetching Tropical Cyclone HELENE-24 (1001101) … 1 countries
Fetching Tropical Cyclone BERYL-24 (1001067) … 13 countries
Fetching Tropical Cyclone ALBERTO-24 (1001066) … 4 countries
Fetching Tropical Cyclone OTIS-23 (1001028) … 1 countries
Fetching Tropical Cyclone NORMA-23 (1001024) … 1 countries
Fetching Tropical Cyclone LIDIA-23 (1001019) … 1 countries
Fetching Tropical Cyclone IDALIA-23 (1001000) … 5 countries
Fetching Tropical Cyclone FRANKLIN-23 (1000996) … 4 countries
Fetching Tropical Cyclone HILARY-23 (1000993) … 2 countries
Fetching Tropical Cyclone LISA-22 (1000944) 

In [11]:
df.head()

,storm_name,event_id,episode_id,source,from_date,alert_level,iso3,country_name,pop_34kt,pop_64kt
0,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,CAN,Canada,581439.0,1813.0
1,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,BMU,Bermuda,16570.0,NaN
2,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,USA,United States,7959.0,NaN
3,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,SPM,St. Pierre & Miquelon,5676.0,NaN
4,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,BHS,The Bahamas,14029.0,2838.0


## 3. Save output results to Azure

In [ ]:
stratus.upload_csv_to_blob(df, "ds-cyclone-exposure/gdacs_historical_national_exposure.csv")